In [2]:
import torch
import torch.nn as nn
from tqdm import tqdm
from torchvision import transforms

In [3]:
def sinusoidal_enc(t, output_dim, x: torch.Tensor):
    batch_size = t.shape[0]
    device = x.device 

    v = torch.zeros(batch_size, output_dim, device=device)

    i = torch.arange(0, output_dim, 2, device=device)
    div_term = 10000 ** (i / output_dim)

    t_expanded = t.unsqueeze(1)

    v[:, 0::2] = torch.sin(t_expanded / div_term)
    v[:, 1::2] = torch.cos(t_expanded / div_term)

    return v

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
        )

        self.mlp = nn.Sequential(
            nn.Linear(t_dim, in_ch), nn.ReLU(), nn.Linear(in_ch, in_ch)
        )

    def forward(self, x, t_enc):
        B, C, _, _ = x.shape
        t_enc = self.mlp(t_enc)
        t_enc = t_enc.view(B, C, 1, 1)
        return self.convs(x + t_enc)

# 1. Label Embedder 구현

In [5]:
class LabelEmbedder(nn.Module):
    def __init__(self, num_classes, time_dim, dropout_prob=0.1):
        super().__init__()
        # FIX: t_dim -> time_dim (파라미터 이름 불일치로 NameError 나던 부분)
        self.embedding = nn.Embedding(num_classes + 1, time_dim)
        self.num_classes = num_classes
        self.dropout_prob = dropout_prob

    def token_drop(self, labels):
        drop_mask = torch.rand(labels.shape[0], device=labels.device) < self.dropout_prob
        labels = torch.where(drop_mask, self.num_classes, labels)
        return labels

    def forward(self, y, train=True):
        if train and self.dropout_prob > 0:
            y = self.token_drop(y)
        return self.embedding(y)


# 클래스 레이블 y를 t_dim 크기 벡터로 변환
# Classifier-Free Guidance를 위해 num_classes+1개를 만들고,
# 마지막 인덱스 (num_classes) 를 '레이블 없음(null)' 토큰으로 사용

# 2. UNet에 Label Embedder 연결 (CFG용)

In [6]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, t_dim=1000, num_classes=None):
        super().__init__()
        self.t_dim = t_dim

        # --- num_classes가 주어지면 LabelEmbedder 생성 ---
        self.num_classes = num_classes
        self.label_embedder = LabelEmbedder(num_classes, t_dim) if num_classes is not None else None
        # -------------------------------------------------------------

        enc_in_chs = [in_ch, 32, 64, 128, 256]

        self.encs = nn.ModuleList(
            [
                ConvBlock(enc_in_chs[i], enc_in_chs[i + 1], t_dim)
                for i in range(len(enc_in_chs) - 1)
            ]
        )
        dec_in_chs = [256, 128, 64, 32]
        self.upconvs = nn.ModuleList(
            [
                nn.ConvTranspose2d(dec_in_chs[i], dec_in_chs[i + 1], 2, 2)
                for i in range(len(dec_in_chs) - 1)
            ]
        )
        self.decs = nn.ModuleList(
            [
                ConvBlock(dec_in_chs[i], dec_in_chs[i + 1], t_dim)
                for i in range(len(dec_in_chs) - 1)
            ]
        )
        self.last_conv = nn.Conv2d(dec_in_chs[-1], in_ch, 1)
        self.maxpool = nn.MaxPool2d(2)

    def forward(self, x, t, y=None, train=True):
        skip_connections = []
        t_enc = sinusoidal_enc(t, self.t_dim, x)

        # --- 레이블 임베딩을 시간 임베딩에 더해줌 ---
        if self.label_embedder is not None and y is not None:
            y_enc = self.label_embedder(y, train=train)
            t_enc = t_enc + y_enc
        # -------------------------------------------------------

        for i, enc in enumerate(self.encs):
            x = enc(x, t_enc)
            if i < len(self.encs) - 1:
                skip_connections.append(x)
                x = self.maxpool(x)

        for upconv, dec in zip(self.upconvs, self.decs):
            x = upconv(x)
            skip = skip_connections.pop()
            x = torch.cat((skip, x), dim=1)
            x = dec(x, t_enc)

        return self.last_conv(x)

# 3. Diffuser (forward/backward process + CFG 샘플링)

In [7]:
class Diffuser(nn.Module):
    def __init__(self, n_steps=1000, beta_start=0.0001, beta_end=0.02):
        super().__init__()
        self.n_steps = n_steps

        betas = torch.linspace(beta_start, beta_end, n_steps)
        alphas = 1 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)

        self.register_buffer("betas", betas)
        self.register_buffer("alphas", alphas)
        self.register_buffer("alpha_bars", alpha_bars)

    def diffusion_process(self, x_0, t):
        device = x_0.device
        T = self.n_steps
        assert (t >= 1).all() and (t <= T).all()
        t_idx = t - 1

        alpha_bar = self.alpha_bars[t_idx]
        N = alpha_bar.size(0)
        alpha_bar = alpha_bar.view(N, 1, 1, 1)
        noise = torch.randn_like(x_0, device=device)

        x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * noise

        return x_t, noise

    def backward_process(self, model, x, t, y=None, w=0.0):
        T = self.n_steps
        assert (t >= 1).all() and (t <= T).all()
        t_idx = t - 1

        alpha = self.alphas[t_idx]
        alpha_bar = self.alpha_bars[t_idx]
        alpha_bar_prev = torch.ones_like(alpha_bar)
        mask = t > 1
        alpha_bar_prev[mask] = self.alpha_bars[t_idx[mask] - 1]

        N = alpha.size(0)
        alpha = alpha.view(N, 1, 1, 1)
        alpha_bar = alpha_bar.view(N, 1, 1, 1)
        alpha_bar_prev = alpha_bar_prev.view(N, 1, 1, 1)

        model.eval()
        with torch.no_grad():
            # --- CFG 분기: 조건부 예측과 무조건부(null 라벨) 예측을 섞음 ---
            if y is not None and w > 0:
                eps_cond = model(x, t, y, train=False)
                y_null = torch.full_like(y, model.num_classes)
                eps_uncond = model(x, t, y_null, train=False)
                eps = (1 + w) * eps_cond - w * eps_uncond
            else:
                eps = model(x, t, y, train=False)
            # ---------------------------------------------------------------

            noise = torch.randn_like(x)
            noise[t == 1] = 0

            mu = (x - ((1 - alpha)) / torch.sqrt(1 - alpha_bar) * eps) / torch.sqrt(alpha)
            std = torch.sqrt((1 - alpha) * (1 - alpha_bar_prev) / (1 - alpha_bar))

        return mu + noise * std

    def tensor_to_img(self, x):
        x = x * 255
        x = x.clamp(0, 255)
        x = x.to(torch.uint8).cpu()
        to_pil = transforms.ToPILImage()

        return to_pil(x)

    def sample(self, model, x_shape=(10, 3, 224, 224), y=None, w=0.0):
        B = x_shape[0]

        device = next(model.parameters()).device
        x = torch.randn(x_shape, device=device)

        for i in tqdm(range(self.n_steps, 0, -1)):
            t = torch.tensor([i] * B, device=device, dtype=torch.long)
            x = self.backward_process(model, x, t, y=y, w=w)

        imgs = [self.tensor_to_img(x[b]) for b in range(B)]

        return imgs

# 4. 사용 예시 (동작 확인용, 실제 학습 루프는 별도)

아래 셀은 shape이 잘 맞는지, forward/backward가 에러 없이 도는지 빠르게 확인하는 스모크 테스트
학습 루프에 넣을 때는 `model(x, t, y)`과 같이 `y`만 추가로 넘겨주면 됨

In [8]:
# 스모크 테스트 (작은 이미지 크기로 빠르게 shape만 확인)
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"

    num_classes = 10
    model = UNet(in_ch=3, t_dim=1000, num_classes=num_classes).to(device)
    diffuser = Diffuser(n_steps=1000).to(device)

    x_0 = torch.randn(4, 3, 32, 32, device=device)
    y = torch.randint(0, num_classes, (4,), device=device)
    t = torch.randint(1, 1000 + 1, (4,), device=device)

    x_t, noise = diffuser.diffusion_process(x_0, t)
    pred_noise = model(x_t, t, y)

    print("x_t:", x_t.shape, "pred_noise:", pred_noise.shape)
    assert pred_noise.shape == x_0.shape, "출력 shape이 입력과 달라!"
    print("forward pass OK")

x_t: torch.Size([4, 3, 32, 32]) pred_noise: torch.Size([4, 3, 32, 32])
forward pass OK
